# Dataset Loading

- Load the raw training text file

- Read the entire dataset into memory

- Inspect dataset size and preview content

In [ ]:
with open("traindata.txt",'r',encoding='utf-8') as f:
       text=f.read()


In [ ]:
print("The length of Data set:",len(text))

In [ ]:
print(text[:1000])

In [ ]:
chars=list(set(text))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)

# Clean the Dataset

-Remove punctuation, digits, and symbols

# Keep only:

- alphabets
- spaces
- newline
- Convert text to lowercase

In [ ]:
import re

with open("traindata.txt", "r", encoding="utf-8") as f:
    text = f.read()

# keep alphabets, spaces, and newline
text = re.sub(r"[^A-Za-z\s\n]", " ", text)

# clean extra spaces but keep line structure
text = re.sub(r"[ ]+", " ", text)

# convert to lowercase
text = text.lower()

with open("cleaned.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("cleaning complete")

# Load Cleaned Dataset

`` Load the processed dataset ``
```Check size and preview text ```

In [ ]:
with open("cleaned.txt",'r',encoding='utf-8') as f:
       text2=f.read()

In [ ]:
print(f"Lenght of the data set:{len(text2)} ")

- First 1000 characters

In [ ]:
print(text2[:1000])

## Build Character Vocabulary

- Extract unique characters from dataset
- Compute vocabulary size
- chars → unique characters
- vocab_size → number of tokens in dataset

In [ ]:
chars2=sorted(list(set(text2)))
vocab_size2=len(chars2)
print(''.join(chars2))
print(vocab_size2)

## Encode and Decode Functions
 - Encode text → token IDs
 - Decode token IDs → text

In [ ]:
stri = {ch:i for i, ch in enumerate(chars2)}
it = {i:ch for i, ch in enumerate(chars2)}

encode = lambda s: [stri[c] for c in s]# a function that take a string triverse it and return a integer list
decode = lambda l: ''.join([it[i] for i in l])# a function that convert the 

print(encode("eamon"))
print(decode(encode("eamon")))

## Convert Dataset to Tensor
- Convert encoded tokens to PyTorch tensor

In [ ]:
import torch
data=torch.tensor(encode(text2),dtype=torch.long)
print(data.dtype,data.shape)

In [ ]:
print(data[:1000])

## Train / Validation Split
`` Split dataset to detect overfitting ``
- 90% training
- 10% validation

In [ ]:
# now split the data into train data and validation data
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [ ]:
block_size=18
train_data[:block_size+1]

In [ ]:
a=train_data[:block_size]
b=train_data[1:block_size+1]
for t in range(block_size):
       context=a[:t+1]
       target=b[t]
       print(f"Input:{context},target:{target}")

- Context and Target Example
- Demonstrate next-token prediction

In [ ]:
batch_size=4
block_size=18
torch.manual_seed(1337)
def get_batch(split):
       if split=='train':
              data=train_data
       else:
              data=val_data
       ix = torch.randint(len(data) - block_size, (batch_size,))
       x=torch.stack([data[i:i+block_size]for i in ix])
       y=torch.stack([data[i+1:i+block_size+1]for i in ix])
       return x,y

xb,yb=get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print("--------------------------------------")

for b in range(batch_size):
       for t in range(block_size):
              context=xb[b,:t+1]
              target=yb[b,t]
              print(f"context:{context} and target:{target} ")


### Bigram Language Model
- Predict next character based only on current character
- Uses embedding lookup

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size2)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


In [ ]:
# Adding a Py torch optimizer 
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
# Training loop
batch_size=32
for steps in range(10000):
       xb,yb=get_batch('train')
       logits,loss=m(xb,yb)
       optimizer.zero_grad(set_to_none=None)
       loss.backward()
       optimizer.step()

print(loss.item())

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))

# MATHS in Self attention

In [ ]:
torch.manual_seed(42)
s = torch.tril(torch.ones(3, 3))
s = s / torch.sum(s, 1, keepdim=True)
d = torch.randint(0,10,(3,2)).float()
c = s @ d
print('s=')
print(s)
print('--')
print('d=')
print(d)
print('--')
print('c=')
print(c)

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
# Aim: x[b,t] = mean_{i<=t} x[b,i]
#option 1
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# option 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

In [ ]:
# option 3: using Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


In [ ]:
# option 4 : self attention

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [ ]:
wei[0]

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5
k.var()

In [ ]:
q.var()

In [ ]:
wei.var()

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

In [ ]:
#mean
x[:,0].mean(), x[:,0].std() 

In [ ]:
x[0,:].mean(), x[0,:].std() 